In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches
from joblib import Parallel, delayed
from scipy.stats import hmean
from matplotlib.ticker import PercentFormatter
import re

In [2]:
nifd_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/NIFD/test_cog')
nacc_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/NACC/test_cog')
adni_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/ADNI/test_cog')
ppmi_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/PPMI/test_cog')
brainlat_path = Path('/projectnb/vkolagrp/bellitti/adrd-foundation-model/adrd_simplified_evaluation/results/brainlat/test_cog')

In [3]:
def option_string_to_dict(options):
    pattern = r"([A-Z])\. ([^\n]+)"
    matches = re.findall(pattern, options)
    return {key: value for key, value in matches}

def load_answers(dir_path, dataset_name):
    # load all parquet files from the directory, stack them into a pandas datafame
    # this only reads the participant ID, ground trush answer and the prediction. 
    # Reading only those columns is significantly faster (about 100x) than loading the whole dataframe.
    # Loading everything is very slow because there are extremely long strings (model outputs) in some columns

    fpaths = list(dir_path.rglob('*.parquet'))

    dfs = []

    cols_to_read = ['ID','ground_truth','prediction','ground_truth_text','options']

    for fpath in tqdm(fpaths):

        model = fpath.parent.name.split('-',3)[-1] 
        benchmark = fpath.parent.parent.name.split('_',1)[-1].upper()

        df = pd.read_parquet(fpath,columns=cols_to_read)
    
        df = df.assign(model=model, benchmark=benchmark)

        df['correct'] = (df['ground_truth'] == df['prediction']).astype(int)

        df['prediction_text'] = df.apply(lambda row: option_string_to_dict(row['options']).get(row['prediction'],'invalid'),axis=1)

        dfs.append(df)

    df = pd.concat(dfs)
    df['dataset'] = dataset_name

    # make these columns Categorical
    group_cols = ["dataset", "benchmark", "model", "ground_truth_text", 'prediction_text']
    for col in group_cols:
        df[col] = pd.Categorical(df[col])

    return df


In [4]:
nacc_res = load_answers(nacc_path,dataset_name='NACC')

  0%|          | 0/8 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:15<00:00,  1.94s/it]


In [5]:
blat_res = load_answers(brainlat_path,dataset_name='BrainLat')

100%|██████████| 8/8 [00:00<00:00, 13.80it/s]


In [6]:
ppmi_res = load_answers(ppmi_path,dataset_name='PPMI')

100%|██████████| 8/8 [00:01<00:00,  4.89it/s]


In [7]:
nifd_res = load_answers(nifd_path,dataset_name='NIFD')

100%|██████████| 8/8 [00:00<00:00, 43.49it/s]


In [8]:
adni_res = load_answers(adni_path,dataset_name='ADNI')

100%|██████████| 8/8 [00:00<00:00, 11.21it/s]


In [9]:
nifd_res.sample()

,ID,ground_truth,prediction,ground_truth_text,options,model,benchmark,correct,prediction_text,dataset
8,2_S_0007,C,B,Dementia (DE),A. Mild Cognitive Impairment (MCI)\nB. Normal ...,NACC-3B-OS-SCE,COG,0,Normal Cognition (NC),NIFD


In [10]:
def _process_group(id, group, n_boot, metric_names, seed):
    """
    Helper function for parallel processing.
    Computes bootstrap CIs for a single group.
    """
    dataset, benchmark, model = id
    n_samples = len(group)
    
    if n_samples == 0:
        return []

    unique_labels = group['ground_truth_text'].unique()
    
    # --- Reproducible RNG per worker ---
    # Create a new, independent generator seeded for this group
    rng = np.random.default_rng(seed)
    
    # Pre-generate all bootstrap indices at once
    bootstrap_indices = rng.integers(0, n_samples, size=(n_boot, n_samples))
    
    # Pre-allocate dicts to store results (using lists is fine)
    boot_results = {name: [] for name in ['Mprecision', 'Mrecall', 'Mf1', 'Hprecision','Hrecall','Hf1']}
    
    # Convert to NumPy arrays ONCE per group
    y_true = group["ground_truth_text"].to_numpy()
    y_pred = group["prediction_text"].to_numpy()
    
    for indices in bootstrap_indices:
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]

        # we get the metrics per class, then average them after 
        p, r, f, s = precision_recall_fscore_support(
            y_true_boot,
            y_pred_boot,
            average=None, # Get per-class metrics
            labels=unique_labels,
            zero_division=0,
        )
        
        # Manually compute macro-average and total support
        boot_results["Hprecision"].append(hmean(p))
        boot_results["Hrecall"].append(hmean(r))
        boot_results["Hf1"].append(hmean(f))

        # # Manually compute macro-average and total support
        boot_results["Mprecision"].append(np.mean(p))
        boot_results["Mrecall"].append(np.mean(r))
        boot_results["Mf1"].append(np.mean(f))


    # --- Calculate quantiles ---
    res_list = []
    for metric_name in metric_names:
        # Convert list to array here
        metric_values = np.array(boot_results[metric_name])
        
        # Use np.partition for O(n) percentile calculation
        low_idx = int(0.025 * n_boot)
        med_idx = int(0.5 * n_boot)
        high_idx = int(0.975 * n_boot)
        
        partitioned = np.partition(metric_values, [low_idx, med_idx, high_idx])
        
        res_list.append(
            {
                "dataset": dataset,
                "benchmark": benchmark,
                "model": model,
                "metric": metric_name,
                "median": partitioned[med_idx],
                "low": partitioned[low_idx],
                "high": partitioned[high_idx],
            }
        )
        
    return res_list


def optimized_bootstrap_parallel(df, metric_names, n_boot, seed=None, n_jobs=-1):
    """
    Compute bootstrap confidence intervals in parallel using joblib.
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe
    metric_names : list of str
        List of metric names: ['precision', 'recall', 'f1']
    n_boot : int
        Number of bootstrap samples
    seed : int, optional
        Random seed for reproducible results
    n_jobs : int, optional
        Number of CPU cores to use. -1 means all cores.
    
    Returns
    -------
    pd.DataFrame
        Results with confidence intervals
    """
    
    # --- Setup ---
    # Create a main RNG to generate seeds for each worker
    main_rng = np.random.default_rng(seed)
    
    # Optimization: Convert to categorical for faster groupby
    # We copy to avoid modifying the user's original DataFrame
    df_copy = df.copy()
    group_cols = ["dataset", "benchmark", "model"]
            
    # Get all groups. Must cast to list() to get a concrete list for Parallel
    groups = list(df_copy.groupby(group_cols, observed=True))
    
    # Generate a unique, independent seed for each group
    n_groups = len(groups)
    group_seeds = main_rng.integers(0, 2**32, size=n_groups)
    
    # --- Parallel Execution ---
    results_list_of_lists = Parallel(n_jobs=n_jobs, verbose=0)(
        delayed(_process_group)(
            id, 
            group, 
            n_boot, 
            metric_names, 
            group_seeds[i] # Pass a unique seed to each job
        )
        for i, (id, group) in enumerate(groups)
    )
    
    # --- Collect Results ---
    # Flatten the list of lists returned by Parallel
    final_results = [item for sublist in results_list_of_lists for item in sublist]
    
    return pd.DataFrame(final_results)

In [11]:
metric_names = ['Mprecision', 'Mrecall', 'Mf1', 'Hprecision','Hrecall','Hf1']

In [12]:
nifd_metrics = optimized_bootstrap_parallel(
    df=nifd_res,
    metric_names=metric_names,
    n_boot=1000,
    seed=42,
    n_jobs=32
)

In [13]:
blat_metrics = optimized_bootstrap_parallel(
    df=blat_res,
    metric_names=metric_names,
    n_boot=1000,
    seed=42,
    n_jobs=32
)

In [14]:
adni_metrics = optimized_bootstrap_parallel(
    df=adni_res,
    metric_names=metric_names,
    n_boot=1000,
    seed=42,
    n_jobs=32
)

In [15]:
nacc_metrics = optimized_bootstrap_parallel(
    df=nacc_res,
    metric_names=metric_names,
    n_boot=100,
    seed=42,
    n_jobs=32
)

In [16]:
ppmi_metrics = optimized_bootstrap_parallel(
    df=ppmi_res,
    metric_names=metric_names,
    n_boot=1000,
    seed=42,
    n_jobs=32
)

In [17]:
all_metrics = pd.concat([adni_metrics,nifd_metrics,nacc_metrics,ppmi_metrics, blat_metrics])

In [18]:
all_metrics.sample(10)

,dataset,benchmark,model,metric,median,low,high
26,NACC,COG,NACC-7B-OS,Mf1,0.764173,0.761986,0.765968
16,NIFD,COG,NACC-3B-OS-SCE,Hrecall,0.629643,0.603285,0.652736
24,BrainLat,COG,NACC-7B-OS,Mprecision,0.869461,0.849103,0.888630
27,ADNI,COG,NACC-7B-OS,Hprecision,0.711537,0.700180,0.722943
18,NIFD,COG,NACC-3B-SCE,Mprecision,0.927489,0.911645,0.942037
36,PPMI,COG,Qwen2.5-3B-Instruct,Mprecision,0.432200,0.422328,0.441982
14,PPMI,COG,NACC-3B-OS-SCE,Mf1,0.579564,0.562518,0.597089
10,ADNI,COG,NACC-3B-OS,Hrecall,0.580158,0.563170,0.595510
33,NACC,COG,NACC-7B-OS-SCE,Hprecision,0.688557,0.684189,0.692442
24,ADNI,COG,NACC-7B-OS,Mprecision,0.715380,0.704157,0.727542


In [28]:
import os
sns.set_style("whitegrid")
df = all_metrics[all_metrics['metric'].isin(
    ['Mrecall','Mprecision','Mf1']
    # ['Hrecall','Hprecision','Hf1']
)].copy()
# Abbreviate metric names
abbrev_map = {
    "Mrecall": "Macro Recall",
    "Mprecision": "Macro Precision",
    "Mf1": "Macro F1 Score",
    "Hrecall": "Harmonic Recall",
    "Hprecision": "Harmonic Precision",
    "Hf1": "Harmonic F1",
}
df["metric_abbrev"] = df["metric"].map(abbrev_map)
# Desired model order (left-to-right as listed)
model_map = {
    "Qwen2.5-3B-Instruct":"Q3B",
    "NACC-3B-OS-SCE":"LUNAR",
    "Qwen2.5-7B-Instruct":"Q7B",
}
df["model_abbrev"] = df["model"].map(model_map)
model_order = [
    "Q3B",
    "LUNAR",
    "Q7B"    
]
# Map models to numeric x positions
n = len(model_order)
pos_map = {m: i for i, m in enumerate(model_order)}
df["model_x"] = df["model_abbrev"].map(pos_map)
# Define colors for datasets
datasets = sorted(df["dataset"].unique())
palette = dict(zip(datasets, sns.color_palette("colorblind", n_colors=len(datasets))))

# Define markers for datasets
markers = ['o', 's', '^', 'D', 'v', 'p', '*', 'X']  # circle, square, triangle-up, diamond, triangle-down, pentagon, star, X
marker_map = dict(zip(datasets, markers[:len(datasets)]))

# Get unique benchmarks
benchmarks = df["benchmark"].unique()
# Create a separate figure for each benchmark
for benchmark in benchmarks:
    # Filter data for this benchmark
    df_bench = df[df["benchmark"] == benchmark]
    # Create FacetGrid (taller than wide, metrics as columns)
    g = sns.FacetGrid(
        df_bench,
        col="metric_abbrev",
        hue="dataset",
        sharex=True,
        sharey=True,
        height=3,  # Increased height
        aspect=0.5,  # Reduced aspect ratio (< 1 makes it taller than wide)
        palette=palette,
        col_wrap=3
    )
    # Plot function
    def facet_plot(data, color, **kwargs):
        for dataset in data["dataset"].unique():
            sub = data[data["dataset"] == dataset].sort_values('model_x')
            plt.errorbar(
                x=sub["model_x"],
                y=sub["median"],
                yerr=[sub["median"] - sub["low"], sub["high"] - sub["median"]],
                fmt=marker_map[dataset] + "-",  # Use dataset-specific marker
                ecolor=palette[dataset],
                color=palette[dataset],
                capsize=0,
                markersize=5,  # Increased slightly for better visibility
                label=None,
                zorder=2,
            )
    g.map_dataframe(facet_plot)
    # Build x-tick labels
    xtick_labels = [model_order[i] for i in range(n)]
    # Apply x-ticks to all panels (since they're side-by-side)
    axes = g.axes
    for ax in axes.flat:
        ax.set_xticks(list(range(n)))
        ax.set_xticklabels(xtick_labels, rotation=90, ha="center")
        ax.set_xlabel("Model")
        ax.tick_params(labelleft=True)
        ax.set_ylim(0.35,1.0)
        ax.set_yticks(np.arange(0.4, 1.01, 0.1))
    # Legend inside the figure (upper left)
    handles = [
        plt.Line2D([0], [0], marker=marker_map[d], color=palette[d], linestyle="", 
                   markersize=7, label=d)
        for d in datasets
    ]
    g.fig.legend(
        handles=handles, title="Dataset", bbox_to_anchor=(1.01, 0.75), loc="upper left"
    )
    # Labels and titles
    g.set_axis_labels("", "")
    g.set_titles(template="{col_name}")
    # Remove spines
    for ax in g.axes.flat:
        sns.despine(ax=ax, left=True, bottom=True, right=True, top=True)
    plt.tight_layout()
    # Save figure
    filename = f"../figures/{benchmark}.pdf"
    plt.savefig(filename, bbox_inches="tight", dpi=300)
    print(f"Saved {filename}")
    plt.close()

Saved ../figures/COG.pdf
